<a href="https://colab.research.google.com/github/nejumi/weave-initial-course/blob/main/notebooks/00_setup/00_environment_setup.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 00. 環境セットアップ

本コースを通じて使用する W&B / Weave の接続設定と動作確認を行います。

## 学習内容
- W&B Dedicated Cloud への接続設定（`WANDB_BASE_URL`）
- `weave.init()` によるプロジェクト初期化
- `@weave.op` の最小動作確認


In [ ]:
# ローカル環境: uv sync --all-extras で一括インストール可能
# Colab: 以下のセルで個別インストール
!pip install -q uv
!uv pip install --system -q \
  "weave>=0.51.0" \
  "wandb>=0.19.0" \
  "openai>=1.0.0" \
  "python-dotenv>=1.0.0"

In [ ]:
import os

# Colab / ローカル環境の自動判定
try:
    import google.colab
    IN_COLAB = True
    from google.colab import userdata
    # WANDB_BASE_URL: 値がある場合のみセット（空 / 未設定 → SaaS デフォルト）
    _base_url = (userdata.get("WANDB_BASE_URL") or "").strip()
    if _base_url:
        os.environ["WANDB_BASE_URL"] = _base_url
    os.environ.setdefault("WANDB_API_KEY",  userdata.get("WANDB_API_KEY"))
    os.environ.setdefault("OPENAI_API_KEY", userdata.get("OPENAI_API_KEY"))
    os.environ.setdefault("WANDB_ENTITY",   userdata.get("WANDB_ENTITY"))
    os.environ.setdefault("WANDB_PROJECT",  userdata.get("WANDB_PROJECT") or "weave-course")
    print("Google Colab 環境")
except ImportError:
    IN_COLAB = False
    from dotenv import load_dotenv
    load_dotenv()
    # ローカル: .env に WANDB_BASE_URL= と書かれていた場合も空なら削除
    if not os.environ.get("WANDB_BASE_URL", "").strip():
        os.environ.pop("WANDB_BASE_URL", None)
    print("ローカル環境")

ENTITY  = os.environ["WANDB_ENTITY"]
PROJECT = os.environ.get("WANDB_PROJECT", "weave-course")
_target = os.environ.get("WANDB_BASE_URL", "https://api.wandb.ai (SaaS)")
print(f"Entity : {ENTITY}")
print(f"Project: {PROJECT}")
print(f"W&B URL: {_target}")


## API キーの設定

### Google Colab の場合
左サイドバー 🔑 Secrets に以下を登録してください：

| キー | 値 |
|---|---|
| `WANDB_BASE_URL` | `https://your-org.wandb.io`（Dedicated Cloud の場合） |
| `WANDB_API_KEY` | W&B の API キー |
| `OPENAI_API_KEY` | OpenAI の API キー |
| `WANDB_ENTITY` | W&B のエンティティ名（ユーザー名 or チーム名） |
| `WANDB_PROJECT` | `weave-course`（任意） |

### ローカルの場合
`.env.example` を `.env` にコピーして値を設定してください。


In [ ]:
import wandb
wandb.login()
print("✓ W&B ログイン成功")


In [ ]:
import weave

client = weave.init(f"{ENTITY}/{PROJECT}")
print(f"✓ Weave 初期化成功")
print(f"  Entity : {client.entity}")
print(f"  Project: {client.project}")


In [ ]:
# 動作確認: 最初の @weave.op

@weave.op()
def hello_weave(name: str) -> str:
    return f"Hello, {name}! Weave is working."

result = hello_weave("World")
print(result)
print(f"\n▶ Weave UI でトレースを確認: https://wandb.ai/{ENTITY}/{PROJECT}/weave")


## セットアップ完了 ✅

次のノートブックへ進んでください: **01_basics/01_traces_and_ops.ipynb**
